# LC 743 — Network Delay Time
**Day 48 | Theme: Shortest Path / Dijkstra**

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px">

**Core Insight:** Use Dijkstra from the source node `k`.
Track the shortest distance to every node via a min-heap.
Once all nodes are settled, the answer is the **maximum**
of those shortest distances — the last node to receive the
signal determines the total delay.
If any node is unreachable, return `-1`.

</div>

## Official Problem Statement

You are given a network of `n` nodes, labeled `1` to `n`.
You are also given `times`, a list of travel times as
directed edges `times[i] = (u_i, v_i, w_i)`, where `u_i`
is the source node, `v_i` is the target node, and `w_i`
is the time it takes for a signal to travel from source
to target.

We will send a signal from a given node `k`. Return *the
**minimum** time it takes for all the `n` nodes to receive
the signal*. If it is impossible for all the `n` nodes to
receive the signal, return `-1`.

**Constraints:**
- `1 <= k <= n <= 100`
- `1 <= times.length <= 6000`
- `times[i].length == 3`
- `1 <= u_i, v_i <= n`
- `u_i != v_i`
- `0 <= w_i <= 100`
- All pairs `(u_i, v_i)` are **unique**.

## What This Is Actually Asking

We are broadcasting a signal from node `k` across a
directed, weighted graph.
The signal travels along edges and each edge has a cost
(travel time).
We want every node to receive the signal, so we need the
shortest path from `k` to **each** node.
The total delay is the **longest** of those shortest
paths — we wait until the slowest node is reached.
If some node has no path from `k`, it can never receive
the signal, so we return `-1`.

## Walk Through an Example by Hand

```
times = [[2,1,1],[2,3,1],[3,4,1]], n=4, k=2
```

**Build adjacency list:**
```
2 -> [(1, w=1), (3, w=1)]
3 -> [(4, w=1)]
```

**Init:** `dist = [inf, inf, 0, inf, inf]` (1-indexed)
Heap: `[(0, 2)]`

**Step 1:** Pop `(0, 2)`. dist[2]=0. Neighbours: 1,3.
- Relax 1: 0+1=1 < inf → dist[1]=1, push (1,1)
- Relax 3: 0+1=1 < inf → dist[3]=1, push (1,3)
Heap: `[(1,1),(1,3)]`

**Step 2:** Pop `(1, 1)`. dist[1]=1. No neighbours.
Heap: `[(1,3)]`

**Step 3:** Pop `(1, 3)`. dist[3]=1. Neighbour: 4.
- Relax 4: 1+1=2 < inf → dist[4]=2, push (2,4)
Heap: `[(2,4)]`

**Step 4:** Pop `(2, 4)`. dist[4]=2. No neighbours.
Heap: `[]`

**Result:** max(dist[1..4]) = max(1,0,1,2) = **2**

## The Picture

```
Graph (directed, weighted):

        w=1       w=1
  [2] -------> [1]   [2] -------> [3]
                              |
                           w=1|
                              v
                             [4]

  Start: k=2
```

```
Dijkstra Priority Queue Evolution:

 Iteration | Heap (min at top)     | dist[1..4]
-----------|----------------------|-------------------
   Init    | [(0,2)]              | [inf, inf, 0, inf]
   Pop 2   | [(1,1),(1,3)]        | [1,   inf, 0, inf]
   Pop 1   | [(1,3)]              | [1,   inf, 0, inf]
   Pop 3   | [(2,4)]              | [1,   inf, 0, 2  ]
   Pop 4   | []                   | [1,   inf, 0, 2  ]

  max(dist[1], dist[2], dist[3], dist[4])
= max(1, 0, 1, 2) = 2  ← answer
```

```
Key rule:
  When we pop (d, u) and d > dist[u] → SKIP (stale entry)
  This is the lazy deletion trick — no decrease-key needed.
```

## When To Use This Pattern

- When you need **shortest path from one source** to all
  others, think **Dijkstra with a min-heap**.
- When edge weights are **non-negative** (costs, times,
  distances), think **Dijkstra** (not Bellman-Ford).
- When the answer depends on reaching **all nodes**
  (broadcast, propagation), think **max of shortest
  paths**.
- When a node is **not reachable** and the problem says
  return a sentinel, think **check for inf in dist**.
- When the graph uses **1-indexed nodes**, think **size
  n+1 arrays** or convert to 0-indexed carefully.

## The Approach

Build an adjacency list from the edge list `times`.
Run Dijkstra starting from node `k`: maintain a min-heap
of `(distance, node)` pairs and a `dist` array initialised
to infinity, with `dist[k] = 0`.
Each time we pop the cheapest unvisited node, we relax
its neighbours — if a shorter path is found, push the new
entry onto the heap (stale entries are discarded on pop).
After the heap empties, return `max(dist[1..n])` if no
value is infinity, otherwise return `-1`.

In [ ]:
import heapq
from typing import List
from collections import defaultdict

In [ ]:
def test_harness(func):
    """Run test cases for networkDelayTime."""
    cases = [
        # (times, n, k, expected)
        # Basic example from LeetCode
        (
            [[2,1,1],[2,3,1],[3,4,1]],
            4, 2, 2,
            "Basic 4-node graph"
        ),
        # Single node, source = only node
        (
            [],
            1, 1, 0,
            "Single node network"
        ),
        # Two nodes, edge exists
        (
            [[1,2,1]],
            2, 1, 1,
            "Two nodes, reachable"
        ),
        # Two nodes, no edge from source
        (
            [[2,1,1]],
            2, 1, -1,
            "Two nodes, unreachable"
        ),
        # Source has shortest paths via different routes
        (
            [[1,2,1],[2,3,2],[1,3,4]],
            3, 1, 3,
            "Two routes to node 3"
        ),
    ]

    passed = 0
    for times, n, k, expected, label in cases:
        result = func(times, n, k)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"[{status}] {label}\n"
            f"         got={result}, expected={expected}"
        )

    print(f"\nResults: {passed}/{len(cases)} passed")

In [ ]:
def networkDelayTime(
    times: List[List[int]], n: int, k: int
) -> int:
    """
    Return the minimum time for all n nodes to receive
    a signal sent from node k, or -1 if impossible.

    Strategy: Dijkstra (single-source shortest path).
    - Build adjacency list from times.
    - Min-heap: (dist, node), init with (0, k).
    - dist array init to inf; dist[k] = 0.
    - Pop cheapest, skip if stale (d > dist[u]).
    - Relax neighbours; push updates onto heap.
    - Answer = max(dist[1..n]), or -1 if any is inf.

    Args:
        times: Directed edges [u, v, w].
        n:     Number of nodes (labeled 1..n).
        k:     Source node.

    Returns:
        Minimum time for all nodes to be reached,
        or -1 if unreachable.

    Examples:
        >>> networkDelayTime([[2,1,1],[2,3,1],[3,4,1]],
        ...                  4, 2)
        2
        >>> networkDelayTime([[1,2,1]], 2, 2)
        -1
    """
    # Debug: inspect inputs
    print(f"[DBG] n={n}, k={k}, edges={len(times)}")

    # TODO: Build adjacency list
    # graph = defaultdict(list)
    # for u, v, w in times:
    #     graph[u].append((v, w))
    print(f"[DBG] adjacency list built")

    # TODO: Init dist and heap
    # dist = [float('inf')] * (n + 1)
    # dist[k] = 0
    # heap = [(0, k)]
    print(f"[DBG] heap initialised with source={k}")

    # TODO: Dijkstra main loop
    # while heap:
    #     d, u = heapq.heappop(heap)
    #     if d > dist[u]:  # stale
    #         continue
    #     for v, w in graph[u]:
    #         nd = d + w
    #         if nd < dist[v]:
    #             dist[v] = nd
    #             heapq.heappush(heap, (nd, v))
    print(f"[DBG] heap exhausted")

    # TODO: Compute answer
    # ans = max(dist[1:n+1])
    # return ans if ans < float('inf') else -1
    print(f"[DBG] returning answer")

    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(networkDelayTime)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute Force (BFS/DFS repeat) | O(V * (V+E)) | O(V+E) | Re-explore from source each time |
| Bellman-Ford | O(V * E) | O(V) | Works with negative weights |
| **Dijkstra + Min-Heap** | **O((V+E) log V)** | **O(V+E)** | **Optimal for non-neg weights** |

Where `V = n` (nodes) and `E = len(times)` (edges).

- **Heap push/pop:** O(log V) per operation.
- **Total pushes:** at most O(E) (one per edge relaxation).
- **Space:** adjacency list O(E), dist array O(V),
  heap O(E) worst case.

## Real World Connection

At **Citi**, trade confirmation messages propagate across
a distributed network of services (risk, compliance,
reporting). The "network delay time" maps directly to the
worst-case latency before all downstream systems are
updated — SLA breaches happen when that maximum exceeds
the threshold.
On **AWS**, services like Route 53 and CloudFront use
shortest-path routing to minimise latency between edge
nodes and the origin; Dijkstra-like algorithms underpin
those routing tables.
In **data engineering**, dependency graphs for pipeline
stages (Airflow DAGs, dbt models) can be analysed with
the same pattern: the critical path is the longest
shortest path, and optimising it reduces end-to-end
pipeline completion time.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra